In [ ]:
# =====================================================================
# BOLUM 0 - Ayarlar, veri, dondurulmus bolme ve surum uyumlulugu
# =====================================================================
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from joblib import Parallel, delayed

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, get_scorer
from scipy import stats

RANDOM_STATE = 42
N_SPLITS = 5
ROUND_DEC = 6
EPS = 1e-8

# ---- HESAPLAMA BUTCESI (kullanici tarafindan degistirilebilir) --------
RANDOM_SEARCH_N_ITER = 100
PSO_SWARM_SIZE = 10
PSO_N_ITERATIONS = 10
RUN_MULTIPLE_PSO_RUNS = True
PSO_SEEDS = [42, 52, 62, 72, 82]
N_JOBS = -1

PSO_W_START, PSO_W_END = 0.9, 0.4
PSO_C1, PSO_C2 = 2.0, 2.0
PSO_VELOCITY_RATIO = 0.5          # hiz siniri = oran * (ust - alt)
PSO_COUNT_INITIAL_AS_ITERATION = True   # True -> toplam cagri = swarm * n_iterations

HIGH_FREQ_THRESHOLD = 2.5         # yalnizca tanisal alt grup icin

# ---- Portable project paths ---------------------------------------------
from pathlib import Path

def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "masonry_tower_primary_dataset.xlsx").is_file():
            return candidate
    raise FileNotFoundError(
        "Project root could not be located. Run this notebook from the repository "
        "root or from its code/ directory, and keep the data/ directory unchanged."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAT_FILE = DATA_DIR / "masonry_tower_primary_dataset.xlsx"
# ------------------------------------------------------------------------
SPLIT_FILE = os.path.join(OUTPUT_DIR, "Data_Split_Assignment.xlsx")
MODEL_DIR  = os.path.join(OUTPUT_DIR, "Models")

GRAFIK_KLASORLERI = {
    "opt_comp":  os.path.join(OUTPUT_DIR, "Optimization_Comparison"),
    "rs_analiz": os.path.join(OUTPUT_DIR, "Randomized_Search_Analysis"),
    "pso_conv":  os.path.join(OUTPUT_DIR, "PSO_Convergence"),
    "pso_stab":  os.path.join(OUTPUT_DIR, "PSO_Stability"),
    "avp":       os.path.join(OUTPUT_DIR, "Optimized_Actual_vs_Predicted"),
    "resid":     os.path.join(OUTPUT_DIR, "Optimized_Residual_Plots"),
    "cost":      os.path.join(OUTPUT_DIR, "Computational_Cost"),
}
for k in [OUTPUT_DIR, MODEL_DIR] + list(GRAFIK_KLASORLERI.values()):
    os.makedirs(k, exist_ok=True)

GEO_COLS = ["Height (m)", "Section a (m)", "Section b (m)", "Wall Thickness (m)",
            "Opening z/H", "Opening Ratio x (%)", "Opening Ratio y (%)"]
MAT_COLS = ["E (MPa)", "d (kg/m3)"]
FEATURE_COLS = GEO_COLS + MAT_COLS
TARGETS = ["f1 (Hz)", "f2 (Hz)"]
TARGET_KISA = {"f1 (Hz)": "f1", "f2 (Hz)": "f2"}
MODELLER = ["SVR", "GBR"]
YONTEMLER = ["Baseline", "Randomized Search", "PSO"]
BEKLENEN_DISARIDA = {"G170", "G187", "G183", "G068", "G099", "G076"}

# ---- Grafik stili -----------------------------------------------------
DPI = 300
sns.set_style("white")
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 12, "axes.labelsize": 12,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 9.5,
    "axes.linewidth": 0.9, "savefig.dpi": DPI, "savefig.bbox": "tight",
})
RENK = {"Baseline": "#8C8C8C", "Randomized Search": "#4C72B0", "PSO": "#55A868"}
RENK_VURGU = "#C44E52"


def kaydet(fig, klasor_anahtari, ad):
    """Figuru PNG olarak kaydeder ve bellekten temizler."""
    yol = os.path.join(GRAFIK_KLASORLERI[klasor_anahtari], ad + ".png")
    fig.savefig(yol, dpi=DPI)
    plt.close(fig)
    print("Kaydedildi:", yol)


# ---- Surum uyumlulugu kontrolleri -------------------------------------
def rmse_scorer_sec():
    """neg_root_mean_squared_error varsa onu, yoksa neg_mean_squared_error kullanir."""
    try:
        get_scorer("neg_root_mean_squared_error")
        return "neg_root_mean_squared_error", True
    except Exception:
        # UYUMLULUK DEGISIKLIGI: skorlar karekok alinarak RMSE'ye cevrilecektir.
        return "neg_mean_squared_error", False


def log_uniform(alt, ust):
    """scipy.stats.loguniform yoksa reciprocal kullanir (surum uyumlulugu)."""
    try:
        return stats.loguniform(alt, ust)
    except AttributeError:
        return stats.reciprocal(alt, ust)


def kare_hata_loss_adi():
    """'squared_error' desteklenmiyorsa uyumluluk icin 'ls' dondurur."""
    X_kucuk = np.arange(20, dtype=float).reshape(-1, 2)
    y_kucuk = np.arange(10, dtype=float)
    for isim in ["squared_error", "ls"]:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                GradientBoostingRegressor(loss=isim, n_estimators=2).fit(X_kucuk, y_kucuk)
            return isim
        except Exception:
            continue
    raise RuntimeError("Uygun GBR kare hata loss adi bulunamadi.")


SCORING, SCORER_RMSE_MI = rmse_scorer_sec()
GBR_KARE_LOSS = kare_hata_loss_adi()
print(f"Scoring: {SCORING} (dogrudan RMSE mi: {SCORER_RMSE_MI})")
print(f"GBR kare hata loss adi: {GBR_KARE_LOSS}")


def skoru_rmse_yap(skor):
    """sklearn negatif skorunu pozitif RMSE'ye cevirir."""
    return -skor if SCORER_RMSE_MI else float(np.sqrt(-skor))


# ---- Veri ve dondurulmus bolme ---------------------------------------
def geometry_id_olustur(veri, geo_cols=GEO_COLS, ndec=ROUND_DEC):
    """Onceki asamalarla birebir ayni Geometry_ID uretimi."""
    anahtar = veri[geo_cols].round(ndec).astype(str).agg("|".join, axis=1)
    esleme = {k: f"G{i+1:03d}" for i, k in enumerate(pd.unique(anahtar))}
    return anahtar.map(esleme)


def dondurulmus_bolmeyi_yukle(veri):
    """Data_Split_Assignment.xlsx dosyasindaki Split ve CV_Fold atamalarini okur."""
    if not os.path.exists(SPLIT_FILE):
        raise FileNotFoundError(
            f"Bolme dosyasi bulunamadi: {SPLIT_FILE}. Asama 3 kodunu calistirin. "
            "Bu kod YENI bir bolme uretmez.")
    atama = pd.read_excel(SPLIT_FILE, sheet_name="Row_assignment").sort_values("Row_index")
    if len(atama) != len(veri):
        raise ValueError(f"Satir sayisi uyusmuyor: veri={len(veri)}, bolme={len(atama)}")
    if not (atama["Geometry_ID"].values == veri["Geometry_ID"].values).all():
        raise ValueError("Geometry_ID sirasi bolme dosyasiyla uyusmuyor.")

    veri = veri.copy()
    veri["Split"] = atama["Split"].values
    veri["CV_Fold"] = atama["CV_Fold"].values

    for ad, bek, goz in [("Train records", 912, int((veri["Split"] == "Train").sum())),
                         ("Test records", 234, int((veri["Split"] == "Test").sum())),
                         ("Train geometries", 152,
                          veri.loc[veri["Split"] == "Train", "Geometry_ID"].nunique()),
                         ("Test geometries", 39,
                          veri.loc[veri["Split"] == "Test", "Geometry_ID"].nunique())]:
        if bek != goz:
            raise ValueError(f"Bolme yapisi farkli -> {ad}: beklenen {bek}, gozlenen {goz}")

    ortak = (set(veri.loc[veri["Split"] == "Train", "Geometry_ID"]) &
             set(veri.loc[veri["Split"] == "Test", "Geometry_ID"]))
    if ortak:
        raise ValueError(f"VERI SIZINTISI: {len(ortak)} geometri hem egitimde hem testte.")
    print("Dondurulmus bolme yuklendi ve dogrulandi (912/234 kayit, 152/39 geometri).")
    return veri


df = pd.read_excel(MAT_FILE, sheet_name=0)
df["Geometry_ID"] = geometry_id_olustur(df)
df = dondurulmus_bolmeyi_yukle(df)

train_mask = (df["Split"] == "Train").values
test_mask = (df["Split"] == "Test").values

X_train = df.loc[train_mask, FEATURE_COLS].reset_index(drop=True)
X_test = df.loc[test_mask, FEATURE_COLS].reset_index(drop=True)
geo_train = df.loc[train_mask, "Geometry_ID"].reset_index(drop=True)
geo_test = df.loc[test_mask, "Geometry_ID"].reset_index(drop=True)
fold_train = df.loc[train_mask, "CV_Fold"].astype(int).reset_index(drop=True)
y_train = {t: df.loc[train_mask, t].reset_index(drop=True) for t in TARGETS}
y_test = {t: df.loc[test_mask, t].reset_index(drop=True) for t in TARGETS}

if "Geometry_ID" in X_train.columns:
    raise ValueError("HATA: Geometry_ID ozellik matrisinde bulunuyor.")

# ---- Test alt grup maskeleri (optimizasyonda KULLANILMAZ) ------------
disarida_mask = np.zeros(len(X_test), dtype=bool)
for kol in FEATURE_COLS:
    disarida_mask |= (X_test[kol] < X_train[kol].min()).values
    disarida_mask |= (X_test[kol] > X_train[kol].max()).values

bulunan_disarida = set(geo_test[disarida_mask].unique())
print(f"Egitim araligi disindaki test kayitlari: {int(disarida_mask.sum())} / {len(X_test)}")
if bulunan_disarida != BEKLENEN_DISARIDA:
    print("UYARI: Otomatik belirlenen geometri kumesi beklenenden farkli!")
    print("  Bulunan :", sorted(bulunan_disarida))
    print("  Beklenen:", sorted(BEKLENEN_DISARIDA))

yuksek_frekans_mask = {t: (y_test[t].values > HIGH_FREQ_THRESHOLD) for t in TARGETS}

In [ ]:
# =====================================================================
# BOLUM 1 - Sabit GroupKFold katlarinin olusturulmasi ve dogrulanmasi
# Bu katlar Randomized Search, PSO, SVR, GBR, f1 ve f2 icin AYNEN kullanilir.
# =====================================================================

def sabit_cv_katlarini_olustur():
    """Dondurulmus CV_Fold sutunundan (train_idx, val_idx) listesi uretir."""
    splits, kontrol = [], []
    for fold in range(1, N_SPLITS + 1):
        val_idx = np.where(fold_train.values == fold)[0]
        tr_idx = np.where(fold_train.values != fold)[0]
        ortak = set(geo_train.iloc[tr_idx]) & set(geo_train.iloc[val_idx])
        if len(ortak) != 0:
            raise ValueError(f"VERI SIZINTISI: Fold {fold} icinde {len(ortak)} ortak geometri.")
        splits.append((tr_idx, val_idx))
        kontrol.append({
            "Fold": fold,
            "Train_samples": len(tr_idx), "Validation_samples": len(val_idx),
            "Train_geometries": geo_train.iloc[tr_idx].nunique(),
            "Validation_geometries": geo_train.iloc[val_idx].nunique(),
            "Common_geometries": 0,
        })
    return splits, pd.DataFrame(kontrol)


CV_SPLITS, CV_KONTROL = sabit_cv_katlarini_olustur()
print("\n--- Sabit CV katlari ---")
print(CV_KONTROL.to_string(index=False))

In [ ]:
# =====================================================================
# BOLUM 2 - Model fabrikasi, arama uzaylari ve PSO kodlama/cozme
# =====================================================================

def pipeline_olustur(model_adi):
    """SVR icin StandardScaler -> SVR, GBR icin yalnizca model."""
    if model_adi == "SVR":
        return Pipeline([("scaler", StandardScaler()),
                         ("model", SVR(kernel="rbf", shrinking=True, cache_size=1000))])
    if model_adi == "GBR":
        return Pipeline([("model", GradientBoostingRegressor(random_state=RANDOM_STATE))])
    raise ValueError(f"Bilinmeyen model: {model_adi}")


def baseline_parametreleri(model_adi):
    """Asama 4'teki temel ayarlar (karsilastirma referansi)."""
    if model_adi == "SVR":
        return {"model__C": 1.0, "model__gamma": "scale", "model__epsilon": 0.1}
    return {}   # GBR: varsayilan parametreler


# ---- Randomized Search arama uzaylari --------------------------------
GBR_MAX_FEATURES = [None, "sqrt", "log2", 0.5, 0.7, 1.0]
GBR_LOSS = [GBR_KARE_LOSS, "huber"]

ARAMA_UZAYI = {
    "SVR": {
        "model__C": log_uniform(1e-1, 1e4),
        "model__gamma": log_uniform(1e-5, 1e1),
        "model__epsilon": log_uniform(1e-3, 0.30),
    },
    "GBR": {
        "model__n_estimators": stats.randint(100, 1001),
        "model__learning_rate": log_uniform(0.005, 0.20),
        "model__max_depth": stats.randint(2, 6),
        "model__min_samples_split": stats.randint(2, 21),
        "model__min_samples_leaf": stats.randint(1, 11),
        "model__subsample": stats.uniform(loc=0.60, scale=0.40),
        "model__max_features": GBR_MAX_FEATURES,
        "model__loss": GBR_LOSS,
    },
}

# ---- PSO arama uzaylari (ayni sinirlar, surekli temsil) ---------------
def svr_coz(pozisyon):
    """PSO konumunu SVR hiperparametrelerine cevirir (log10 uzayindan)."""
    return {"model__C": float(10.0 ** pozisyon[0]),
            "model__gamma": float(10.0 ** pozisyon[1]),
            "model__epsilon": float(10.0 ** pozisyon[2])}


def gbr_coz(pozisyon):
    """PSO konumunu GBR hiperparametrelerine cevirir (deterministik yuvarlama)."""
    mf_idx = int(np.clip(round(pozisyon[6]), 0, len(GBR_MAX_FEATURES) - 1))
    loss_idx = int(np.clip(round(pozisyon[7]), 0, len(GBR_LOSS) - 1))
    return {
        "model__n_estimators": int(np.clip(round(pozisyon[0]), 100, 1000)),
        "model__learning_rate": float(10.0 ** pozisyon[1]),
        "model__max_depth": int(np.clip(round(pozisyon[2]), 2, 5)),
        "model__min_samples_split": int(np.clip(round(pozisyon[3]), 2, 20)),
        "model__min_samples_leaf": int(np.clip(round(pozisyon[4]), 1, 10)),
        "model__subsample": float(np.clip(pozisyon[5], 0.60, 1.00)),
        "model__max_features": GBR_MAX_FEATURES[mf_idx],
        "model__loss": GBR_LOSS[loss_idx],
    }


PSO_UZAYI = {
    "SVR": {
        "bounds": [(-1.0, 4.0),                              # log10(C)
                   (-5.0, 1.0),                              # log10(gamma)
                   (float(np.log10(0.001)), float(np.log10(0.30)))],   # log10(epsilon)
        "coz": svr_coz,
        "isimler": ["log10_C", "log10_gamma", "log10_epsilon"],
    },
    "GBR": {
        "bounds": [(100.0, 1000.0),
                   (float(np.log10(0.005)), float(np.log10(0.20))),
                   (2.0, 5.0), (2.0, 20.0), (1.0, 10.0), (0.60, 1.00),
                   (0.0, float(len(GBR_MAX_FEATURES) - 1)),
                   (0.0, float(len(GBR_LOSS) - 1))],
        "coz": gbr_coz,
        "isimler": ["n_estimators", "log10_learning_rate", "max_depth",
                    "min_samples_split", "min_samples_leaf", "subsample",
                    "max_features_index", "loss_index"],
    },
}

In [ ]:
# =====================================================================
# BOLUM 3 - Ortak amac fonksiyonu: bes sabit fold uzerinde ortalama CV RMSE
# Test kumesi bu fonksiyonda HICBIR bicimde kullanilmaz.
# =====================================================================

AMAC_CACHE = {}      # (model, hedef) -> {param_key: (mean_rmse, fold_rmse_listesi)}
AMAC_SAYAC = {}      # (model, hedef) -> sayaclar


def _sayac_al(model_adi, hedef):
    anahtar = (model_adi, hedef)
    if anahtar not in AMAC_SAYAC:
        AMAC_SAYAC[anahtar] = {"total_calls": 0, "unique_evaluations": 0,
                               "cached_evaluations": 0, "model_fits": 0}
    return AMAC_SAYAC[anahtar]


def _param_anahtari(parametreler):
    """Onbellek icin deterministik ve hashlenebilir anahtar."""
    return tuple(sorted((k, str(v)) for k, v in parametreler.items()))


def _tek_fold_rmse(model_adi, hedef_degerleri, parametreler, tr_idx, va_idx):
    """Bir fold icin modeli egitir ve validation RMSE dondurur."""
    pipe = pipeline_olustur(model_adi)
    if parametreler:
        pipe.set_params(**parametreler)
    pipe.fit(X_train.iloc[tr_idx], hedef_degerleri.iloc[tr_idx])
    tahmin = pipe.predict(X_train.iloc[va_idx])
    return float(np.sqrt(mean_squared_error(hedef_degerleri.iloc[va_idx], tahmin)))


def amac_fonksiyonu(model_adi, hedef, parametreler):
    """Objective = bes GroupKFold katindaki ortalama validation RMSE.
    Dondurur: (ortalama_rmse, fold_rmse_listesi, onbellekten_mi)."""
    sayac = _sayac_al(model_adi, hedef)
    sayac["total_calls"] += 1

    onbellek = AMAC_CACHE.setdefault((model_adi, hedef), {})
    anahtar = _param_anahtari(parametreler)
    if anahtar in onbellek:
        sayac["cached_evaluations"] += 1
        ortalama, fold_listesi = onbellek[anahtar]
        return ortalama, fold_listesi, True

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fold_degerleri = Parallel(n_jobs=N_JOBS)(
            delayed(_tek_fold_rmse)(model_adi, y_train[hedef], parametreler, tr, va)
            for tr, va in CV_SPLITS)

    ortalama = float(np.mean(fold_degerleri))
    onbellek[anahtar] = (ortalama, [float(v) for v in fold_degerleri])
    sayac["unique_evaluations"] += 1
    sayac["model_fits"] += len(CV_SPLITS)
    return ortalama, [float(v) for v in fold_degerleri], False


def cv_metriklerini_hesapla(model_adi, hedef, parametreler):
    """En iyi parametrelerle ayni fold'larda R2, RMSE, MAE'yi yeniden hesaplar."""
    satirlar = []
    for fold, (tr, va) in enumerate(CV_SPLITS, start=1):
        pipe = pipeline_olustur(model_adi)
        if parametreler:
            pipe.set_params(**parametreler)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pipe.fit(X_train.iloc[tr], y_train[hedef].iloc[tr])
        tahmin = pipe.predict(X_train.iloc[va])
        gercek = y_train[hedef].iloc[va]
        satirlar.append({
            "Fold": fold,
            "R2": r2_score(gercek, tahmin),
            "RMSE": float(np.sqrt(mean_squared_error(gercek, tahmin))),
            "MAE": mean_absolute_error(gercek, tahmin),
        })
    tablo = pd.DataFrame(satirlar)
    ozet = {
        "CV_R2_mean": tablo["R2"].mean(), "CV_R2_std": tablo["R2"].std(),
        "CV_RMSE_mean": tablo["RMSE"].mean(), "CV_RMSE_std": tablo["RMSE"].std(),
        "CV_MAE_mean": tablo["MAE"].mean(), "CV_MAE_std": tablo["MAE"].std(),
    }
    return ozet, tablo

In [ ]:
# =====================================================================
# BOLUM 4 - Randomized Search (klasik referans yontem)
# =====================================================================

def randomized_search_calistir(model_adi, hedef):
    """Sabit CV katlariyla RandomizedSearchCV calistirir ve tum sonuclari dondurur."""
    pipe = pipeline_olustur(model_adi)
    arama = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=ARAMA_UZAYI[model_adi],
        n_iter=RANDOM_SEARCH_N_ITER,
        cv=CV_SPLITS,                 # onceden olusturulmus sabit katlar
        scoring=SCORING,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        refit=True,
        return_train_score=True,
        error_score=np.nan,
    )

    baslangic = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # groups, API tutarliligi icin geciriliyor; katlar zaten sabittir.
        arama.fit(X_train, y_train[hedef], groups=geo_train)
    sure = time.perf_counter() - baslangic

    sonuc = pd.DataFrame(arama.cv_results_)
    # Negatif sklearn skorlarini pozitif RMSE'ye cevir (split bazinda, kesin donusum)
    val_kolonlari = [f"split{i}_test_score" for i in range(N_SPLITS)]
    tr_kolonlari = [f"split{i}_train_score" for i in range(N_SPLITS)]
    val_rmse = sonuc[val_kolonlari].applymap(skoru_rmse_yap)
    tr_rmse = sonuc[tr_kolonlari].applymap(skoru_rmse_yap)

    tablo = pd.DataFrame({
        "Target": TARGET_KISA[hedef],
        "Model": model_adi,
        "Search_method": "Randomized Search",
        "Parameter_combination": [str(p) for p in sonuc["params"]],
        "Mean_train_RMSE": tr_rmse.mean(axis=1).values,
        "Std_train_RMSE": tr_rmse.std(axis=1, ddof=1).values,
        "Mean_validation_RMSE": val_rmse.mean(axis=1).values,
        "Std_validation_RMSE": val_rmse.std(axis=1, ddof=1).values,
        "Mean_fit_time": sonuc["mean_fit_time"].values,
        "Mean_score_time": sonuc["mean_score_time"].values,
    })
    for i in range(N_SPLITS):
        tablo[f"Fold{i+1}_validation_RMSE"] = val_rmse.iloc[:, i].values
    # Her bir parametrenin ayri sutun olarak eklenmesi (grafikler icin)
    for param_adi in ARAMA_UZAYI[model_adi].keys():
        tablo[param_adi] = [p.get(param_adi) for p in sonuc["params"]]

    tablo["Validation_rank"] = tablo["Mean_validation_RMSE"].rank(method="min").astype(int)
    tablo = tablo.sort_values("Validation_rank").reset_index(drop=True)
    tablo["Total_candidate_count"] = len(sonuc)
    tablo["Total_fit_count"] = len(sonuc) * N_SPLITS

    en_iyi_satir = tablo.iloc[0]
    en_iyi_parametreler = {k: sonuc["params"][int(en_iyi_satir.name)][k]
                           for k in ARAMA_UZAYI[model_adi].keys()} \
        if False else dict(arama.best_params_)

    maliyet = {
        "Target": TARGET_KISA[hedef], "Model": model_adi,
        "Optimization_method": "Randomized Search",
        "Total_candidate_evaluations": int(len(sonuc)),
        "Unique_candidate_evaluations": int(len(sonuc)),
        "Cached_evaluations": 0,
        "Total_model_fits": int(len(sonuc) * N_SPLITS),
        "Refit_count": 1,
        "Runtime_seconds": float(sure),
        "Best_CV_RMSE_from_search": float(tablo["Mean_validation_RMSE"].min()),
    }
    return tablo, en_iyi_parametreler, maliyet


rs_tablolari, rs_en_iyi, rs_maliyet = [], {}, []
print("\n--- Randomized Search basliyor ---")
for hedef in TARGETS:
    for model_adi in MODELLER:
        tablo, en_iyi, maliyet = randomized_search_calistir(model_adi, hedef)
        rs_tablolari.append(tablo)
        rs_en_iyi[(model_adi, hedef)] = en_iyi
        rs_maliyet.append(maliyet)
        print(f"  {TARGET_KISA[hedef]} | {model_adi} | en iyi CV RMSE = "
              f"{maliyet['Best_CV_RMSE_from_search']:.5f} | sure = {maliyet['Runtime_seconds']:.1f} s")

rs_tum = pd.concat(rs_tablolari, ignore_index=True)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Randomized_Search_Results.xlsx"),
                    engine="openpyxl") as writer:
    for hedef in TARGETS:
        for model_adi in MODELLER:
            alt = rs_tum[(rs_tum["Target"] == TARGET_KISA[hedef]) &
                         (rs_tum["Model"] == model_adi)]
            alt.round(6).to_excel(writer,
                                  sheet_name=f"{TARGET_KISA[hedef]}_{model_adi}", index=False)
    pd.DataFrame([{"Target": TARGET_KISA[h], "Model": m, "Best_parameters": str(rs_en_iyi[(m, h)])}
                  for h in TARGETS for m in MODELLER]).to_excel(
        writer, sheet_name="Best_parameters", index=False)

In [ ]:
# =====================================================================
# BOLUM 5 - Particle Swarm Optimization (acik, modüler uygulama)
# =====================================================================

def pso_calistir(model_adi, hedef, seed,
                 swarm_size=PSO_SWARM_SIZE, n_iterations=PSO_N_ITERATIONS):
    """Standart PSO. Konum, hiz, pbest, gbest, atalet, sinir kirpma ve
    yakinsama gecmisi acikca tutulur. Amac: ortalama GroupKFold CV RMSE."""
    uzay = PSO_UZAYI[model_adi]
    sinirlar = np.array(uzay["bounds"], dtype=float)
    alt, ust = sinirlar[:, 0], sinirlar[:, 1]
    boyut = len(alt)
    coz = uzay["coz"]

    rng = np.random.default_rng(seed)
    hiz_siniri = PSO_VELOCITY_RATIO * (ust - alt)

    # Baslangic surusu
    konum = rng.uniform(alt, ust, size=(swarm_size, boyut))
    hiz = rng.uniform(-hiz_siniri, hiz_siniri, size=(swarm_size, boyut))

    pbest_konum = konum.copy()
    pbest_deger = np.full(swarm_size, np.inf)
    gbest_konum = konum[0].copy()
    gbest_deger = np.inf

    gecmis, parcacik_kayitlari = [], []
    baslangic = time.perf_counter()

    def suru_degerlendir(iterasyon):
        """Tum parcaciklari degerlendirir, pbest ve gbest gunceller."""
        nonlocal gbest_deger, gbest_konum
        degerler = np.empty(swarm_size)
        for p in range(swarm_size):
            parametreler = coz(konum[p])
            t0 = time.perf_counter()
            ortalama, fold_listesi, onbellekten = amac_fonksiyonu(model_adi, hedef, parametreler)
            gecen = time.perf_counter() - t0
            degerler[p] = ortalama

            if ortalama < pbest_deger[p]:
                pbest_deger[p] = ortalama
                pbest_konum[p] = konum[p].copy()
            if ortalama < gbest_deger:
                gbest_deger = ortalama
                gbest_konum = konum[p].copy()

            kayit = {"Target": TARGET_KISA[hedef], "Model": model_adi, "Seed": seed,
                     "Iteration": iterasyon, "Particle_ID": p,
                     "Mean_CV_RMSE": ortalama, "Evaluation_time_s": gecen,
                     "Cache_status": "cached" if onbellekten else "evaluated"}
            kayit.update({k.replace("model__", ""): v for k, v in parametreler.items()})
            for i, deger in enumerate(fold_listesi):
                kayit[f"Fold{i+1}_RMSE"] = deger
            parcacik_kayitlari.append(kayit)
        return degerler

    # Iterasyon 0: baslangic surusunun degerlendirilmesi
    degerler = suru_degerlendir(0)
    gecmis.append({"Target": TARGET_KISA[hedef], "Model": model_adi, "Seed": seed,
                   "Iteration": 0, "Global_best_CV_RMSE": gbest_deger,
                   "Iteration_best_CV_RMSE": float(degerler.min()),
                   "Mean_swarm_CV_RMSE": float(degerler.mean()),
                   "Std_swarm_CV_RMSE": float(degerler.std(ddof=1))})

    # Butce esitligi: baslangic degerlendirmesi birinci iterasyon sayilirsa
    # toplam cagri = swarm_size * n_iterations olur.
    guncelleme_sayisi = (n_iterations - 1) if PSO_COUNT_INITIAL_AS_ITERATION else n_iterations
    guncelleme_sayisi = max(guncelleme_sayisi, 0)

    for it in range(1, guncelleme_sayisi + 1):
        # Atalet agirligi 0.9 -> 0.4 dogrusal azalis
        w = (PSO_W_START if guncelleme_sayisi <= 1 else
             PSO_W_START - (PSO_W_START - PSO_W_END) * (it - 1) / (guncelleme_sayisi - 1)) \
            if guncelleme_sayisi > 1 else PSO_W_START

        r1 = rng.random((swarm_size, boyut))
        r2 = rng.random((swarm_size, boyut))
        hiz = (w * hiz
               + PSO_C1 * r1 * (pbest_konum - konum)
               + PSO_C2 * r2 * (gbest_konum - konum))
        hiz = np.clip(hiz, -hiz_siniri, hiz_siniri)      # hiz sinirlandirmasi
        konum = np.clip(konum + hiz, alt, ust)           # sinir disi konumlari kirp

        degerler = suru_degerlendir(it)
        gecmis.append({"Target": TARGET_KISA[hedef], "Model": model_adi, "Seed": seed,
                       "Iteration": it, "Global_best_CV_RMSE": gbest_deger,
                       "Iteration_best_CV_RMSE": float(degerler.min()),
                       "Mean_swarm_CV_RMSE": float(degerler.mean()),
                       "Std_swarm_CV_RMSE": float(degerler.std(ddof=1))})

    sure = time.perf_counter() - baslangic
    return {
        "target": hedef, "model": model_adi, "seed": seed,
        "best_position": gbest_konum, "best_params": coz(gbest_konum),
        "best_cv_rmse": float(gbest_deger),
        "history": pd.DataFrame(gecmis),
        "particles": pd.DataFrame(parcacik_kayitlari),
        "runtime_s": float(sure),
        "total_calls": swarm_size * (guncelleme_sayisi + 1),
    }


# ---- PSO koşuları -----------------------------------------------------
pso_kosulari, pso_gecmisleri, pso_parcaciklari, pso_maliyet = {}, [], [], []
kullanilacak_seedler = PSO_SEEDS if RUN_MULTIPLE_PSO_RUNS else [RANDOM_STATE]

print("\n--- PSO basliyor ---")
for hedef in TARGETS:
    for model_adi in MODELLER:
        sayac_baslangic = dict(_sayac_al(model_adi, hedef))
        for seed in kullanilacak_seedler:
            sonuc = pso_calistir(model_adi, hedef, seed)
            pso_kosulari[(model_adi, hedef, seed)] = sonuc
            pso_gecmisleri.append(sonuc["history"])
            pso_parcaciklari.append(sonuc["particles"])
            print(f"  {TARGET_KISA[hedef]} | {model_adi} | seed {seed} | "
                  f"en iyi CV RMSE = {sonuc['best_cv_rmse']:.5f} | "
                  f"sure = {sonuc['runtime_s']:.1f} s")

        # Ana kosu (seed=42) icin maliyet: kendi kayitlarindan hesaplanir
        ana = pso_kosulari[(model_adi, hedef, RANDOM_STATE)]
        ana_p = ana["particles"]
        pso_maliyet.append({
            "Target": TARGET_KISA[hedef], "Model": model_adi,
            "Optimization_method": "PSO",
            "Total_candidate_evaluations": int(len(ana_p)),
            "Unique_candidate_evaluations": int((ana_p["Cache_status"] == "evaluated").sum()),
            "Cached_evaluations": int((ana_p["Cache_status"] == "cached").sum()),
            "Total_model_fits": int((ana_p["Cache_status"] == "evaluated").sum() * N_SPLITS),
            "Refit_count": 1,
            "Runtime_seconds": ana["runtime_s"],
            "Best_CV_RMSE_from_search": ana["best_cv_rmse"],
        })

pso_gecmis_df = pd.concat(pso_gecmisleri, ignore_index=True)
pso_parcacik_df = pd.concat(pso_parcaciklari, ignore_index=True)

# Ana PSO modeli: YALNIZCA CV RMSE'ye gore secilir (test kumesi kullanilmaz)
pso_en_iyi = {}
for hedef in TARGETS:
    for model_adi in MODELLER:
        adaylar = [pso_kosulari[(model_adi, hedef, s)] for s in kullanilacak_seedler]
        en_iyi_kosu = min(adaylar, key=lambda d: d["best_cv_rmse"])
        pso_en_iyi[(model_adi, hedef)] = en_iyi_kosu

# ---- Kararlilik ozeti -------------------------------------------------
kararlilik_satirlari, kararlilik_ozet = [], []
for hedef in TARGETS:
    for model_adi in MODELLER:
        degerler = []
        for seed in kullanilacak_seedler:
            k = pso_kosulari[(model_adi, hedef, seed)]
            degerler.append(k["best_cv_rmse"])
            kararlilik_satirlari.append({
                "Target": TARGET_KISA[hedef], "Model": model_adi, "Seed": seed,
                "Best_CV_RMSE": k["best_cv_rmse"],
                "Best_parameters": str(k["best_params"]),
                "Runtime_seconds": k["runtime_s"],
                "Total_objective_calls": k["total_calls"],
                "Unique_evaluations": int((k["particles"]["Cache_status"] == "evaluated").sum()),
                "Cached_evaluations": int((k["particles"]["Cache_status"] == "cached").sum()),
            })
        d = np.array(degerler, dtype=float)
        kararlilik_ozet.append({
            "Target": TARGET_KISA[hedef], "Model": model_adi,
            "Number_of_runs": len(d), "Mean_best_CV_RMSE": d.mean(),
            "Std_best_CV_RMSE": d.std(ddof=1) if len(d) > 1 else np.nan,
            "Min_best_CV_RMSE": d.min(), "Median_best_CV_RMSE": float(np.median(d)),
            "Max_best_CV_RMSE": d.max(),
        })

kararlilik_df = pd.DataFrame(kararlilik_satirlari)
kararlilik_ozet_df = pd.DataFrame(kararlilik_ozet)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "PSO_Search_Results.xlsx"),
                    engine="openpyxl") as writer:
    for hedef in TARGETS:
        for model_adi in MODELLER:
            alt = pso_parcacik_df[(pso_parcacik_df["Target"] == TARGET_KISA[hedef]) &
                                  (pso_parcacik_df["Model"] == model_adi)]
            alt.round(6).to_excel(writer,
                                  sheet_name=f"{TARGET_KISA[hedef]}_{model_adi}", index=False)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "PSO_Convergence_Results.xlsx"),
                    engine="openpyxl") as writer:
    for hedef in TARGETS:
        for model_adi in MODELLER:
            alt = pso_gecmis_df[(pso_gecmis_df["Target"] == TARGET_KISA[hedef]) &
                                (pso_gecmis_df["Model"] == model_adi)]
            alt.round(6).to_excel(writer,
                                  sheet_name=f"{TARGET_KISA[hedef]}_{model_adi}", index=False)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "PSO_Stability_Results.xlsx"),
                    engine="openpyxl") as writer:
    kararlilik_df.round(6).to_excel(writer, sheet_name="Runs", index=False)
    kararlilik_ozet_df.round(6).to_excel(writer, sheet_name="Summary", index=False)

In [ ]:
# =====================================================================
# BOLUM 6 - En iyi parametrelerle yeniden CV + bagimsiz test degerlendirmesi
# Test kumesine ILK KEZ burada erisilmektedir.
# =====================================================================

def yontem_parametreleri(yontem, model_adi, hedef):
    """Ilgili yontemin en iyi hiperparametrelerini dondurur."""
    if yontem == "Baseline":
        return baseline_parametreleri(model_adi)
    if yontem == "Randomized Search":
        return rs_en_iyi[(model_adi, hedef)]
    if yontem == "PSO":
        return pso_en_iyi[(model_adi, hedef)]["best_params"]
    raise ValueError(f"Bilinmeyen yontem: {yontem}")


def metrik_seti(gercek, tahmin):
    """R2, RMSE, MAE ve Mean Error dondurur."""
    gercek = np.asarray(gercek, dtype=float)
    tahmin = np.asarray(tahmin, dtype=float)
    return {"R2": r2_score(gercek, tahmin),
            "RMSE": float(np.sqrt(mean_squared_error(gercek, tahmin))),
            "MAE": mean_absolute_error(gercek, tahmin),
            "Mean_Error": float(np.mean(tahmin - gercek))}


def guvenli_yuzde_hata(gercek, tahmin, eps=EPS):
    """Sifira bolunmeye karsi guvenli yuzde hata (%)."""
    gercek = np.asarray(gercek, dtype=float)
    payda = np.where(np.abs(gercek) < eps, np.nan, gercek)
    return 100.0 * (np.asarray(tahmin, dtype=float) - gercek) / payda


cv_ozet_satirlari, cv_fold_satirlari = [], []
tt_satirlari, tahmin_satirlari = [], []
altgrup_satirlari, yuksek_frekans_satirlari = [], []
egitilmis_modeller = {}

print("\n--- Optimize edilmis modellerin degerlendirilmesi ---")
for hedef in TARGETS:
    for model_adi in MODELLER:
        for yontem in YONTEMLER:
            parametreler = yontem_parametreleri(yontem, model_adi, hedef)

            # (a) Ayni sabit fold'larda yeniden CV
            ozet, fold_tablosu = cv_metriklerini_hesapla(model_adi, hedef, parametreler)
            satir = {"Target": TARGET_KISA[hedef], "Model": model_adi,
                     "Optimization_method": yontem}
            satir.update(ozet)
            cv_ozet_satirlari.append(satir)
            fold_tablosu = fold_tablosu.assign(Target=TARGET_KISA[hedef], Model=model_adi,
                                               Optimization_method=yontem)
            cv_fold_satirlari.append(fold_tablosu)

            # (b) Tum egitim kumesinde egitim
            pipe = pipeline_olustur(model_adi)
            if parametreler:
                pipe.set_params(**parametreler)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                pipe.fit(X_train, y_train[hedef])
            egitilmis_modeller[(yontem, model_adi, hedef)] = pipe

            tahmin_tr = pipe.predict(X_train)
            tahmin_te = pipe.predict(X_test)
            m_tr = metrik_seti(y_train[hedef], tahmin_tr)
            m_te = metrik_seti(y_test[hedef], tahmin_te)

            tt_satirlari.append({
                "Target": TARGET_KISA[hedef], "Model": model_adi,
                "Optimization_method": yontem, "Best_parameters": str(parametreler),
                "Train_R2": m_tr["R2"], "Train_RMSE": m_tr["RMSE"], "Train_MAE": m_tr["MAE"],
                "Test_R2": m_te["R2"], "Test_RMSE": m_te["RMSE"], "Test_MAE": m_te["MAE"],
                "Test_Mean_Error": m_te["Mean_Error"],
                "R2_gap_train_minus_test": m_tr["R2"] - m_te["R2"],
            })

            # (c) Tahmin kayitlari
            gercek = y_test[hedef].values
            tahmin_satirlari.append(pd.DataFrame({
                "Geometry_ID": geo_test.values,
                "Optimization_Method": yontem,
                "Model": model_adi,
                "Target": TARGET_KISA[hedef],
                "Actual": gercek,
                "Predicted": tahmin_te,
                "Residual": gercek - tahmin_te,
                "Signed_Error": tahmin_te - gercek,
                "Absolute_Error": np.abs(tahmin_te - gercek),
                "Percentage_Error": guvenli_yuzde_hata(gercek, tahmin_te),
                "Outside_Training_Range": np.where(disarida_mask, "Yes", "No"),
                "High_Frequency_Group": np.where(yuksek_frekans_mask[hedef],
                                                 f"> {HIGH_FREQ_THRESHOLD} Hz",
                                                 f"<= {HIGH_FREQ_THRESHOLD} Hz"),
            }))

            # (d) Tasarim uzayi alt gruplari
            for etiket, maske in [("Inside training range", ~disarida_mask),
                                  ("Outside training range", disarida_mask)]:
                kayit = {"Target": TARGET_KISA[hedef], "Model": model_adi,
                         "Optimization_method": yontem, "Subgroup": etiket,
                         "N_records": int(maske.sum())}
                kayit.update(metrik_seti(gercek[maske], tahmin_te[maske])
                             if maske.sum() > 1 else
                             {"R2": np.nan, "RMSE": np.nan, "MAE": np.nan, "Mean_Error": np.nan})
                kayit.pop("R2", None)   # kucuk alt grupta R2 raporlanmaz
                altgrup_satirlari.append(kayit)

            # (e) Yuksek frekans alt gruplari (yalnizca tanisal)
            for etiket, maske in [(f"<= {HIGH_FREQ_THRESHOLD} Hz", ~yuksek_frekans_mask[hedef]),
                                  (f"> {HIGH_FREQ_THRESHOLD} Hz", yuksek_frekans_mask[hedef])]:
                kayit = {"Target": TARGET_KISA[hedef], "Model": model_adi,
                         "Optimization_method": yontem, "Frequency_group": etiket,
                         "N_records": int(maske.sum())}
                if maske.sum() > 0:
                    m = metrik_seti(gercek[maske], tahmin_te[maske]) if maske.sum() > 1 else \
                        {"RMSE": float(abs(tahmin_te[maske][0] - gercek[maske][0])),
                         "MAE": float(abs(tahmin_te[maske][0] - gercek[maske][0])),
                         "Mean_Error": float(tahmin_te[maske][0] - gercek[maske][0])}
                    kayit.update({"RMSE": m["RMSE"], "MAE": m["MAE"],
                                  "Mean_Error": m["Mean_Error"]})
                else:
                    kayit.update({"RMSE": np.nan, "MAE": np.nan, "Mean_Error": np.nan})
                yuksek_frekans_satirlari.append(kayit)

        print(f"  {TARGET_KISA[hedef]} | {model_adi} | uc yontem tamamlandi")

# ---- Optimize edilmis modellerin joblib olarak kaydedilmesi -----------
for hedef in TARGETS:
    for model_adi in MODELLER:
        for yontem, onek in [("Randomized Search", "RandomSearch"), ("PSO", "PSO")]:
            dosya = f"{onek}_{model_adi}_{TARGET_KISA[hedef]}.joblib"
            joblib.dump(egitilmis_modeller[(yontem, model_adi, hedef)],
                        os.path.join(MODEL_DIR, dosya))
            print("Model kaydedildi:", dosya)

cv_ozet_df = pd.DataFrame(cv_ozet_satirlari)
cv_fold_df = pd.concat(cv_fold_satirlari, ignore_index=True)
tt_df = pd.DataFrame(tt_satirlari)
tahmin_df = pd.concat(tahmin_satirlari, ignore_index=True)
altgrup_df = pd.DataFrame(altgrup_satirlari)
yuksek_frekans_df = pd.DataFrame(yuksek_frekans_satirlari)

In [ ]:
# =====================================================================
# BOLUM 7 - Baseline / Randomized Search / PSO karsilastirmasi ve maliyet
# =====================================================================

def iyilesme_yuzdesi(referans, yeni):
    """(referans - yeni) / referans * 100. Negatif deger kotulesme demektir."""
    if referans is None or np.isnan(referans) or abs(referans) < EPS:
        return np.nan
    return 100.0 * (referans - yeni) / referans


karsilastirma_satirlari = []
for hedef in TARGETS:
    for model_adi in MODELLER:
        kisa = TARGET_KISA[hedef]
        degerler = {}
        for yontem in YONTEMLER:
            cv = cv_ozet_df[(cv_ozet_df["Target"] == kisa) &
                            (cv_ozet_df["Model"] == model_adi) &
                            (cv_ozet_df["Optimization_method"] == yontem)].iloc[0]
            tt = tt_df[(tt_df["Target"] == kisa) &
                       (tt_df["Model"] == model_adi) &
                       (tt_df["Optimization_method"] == yontem)].iloc[0]
            degerler[yontem] = {"cv": cv, "tt": tt}
            karsilastirma_satirlari.append({
                "Target": kisa, "Model": model_adi, "Optimization_method": yontem,
                "CV_R2_mean": cv["CV_R2_mean"], "CV_R2_std": cv["CV_R2_std"],
                "CV_RMSE_mean": cv["CV_RMSE_mean"], "CV_RMSE_std": cv["CV_RMSE_std"],
                "CV_MAE_mean": cv["CV_MAE_mean"], "CV_MAE_std": cv["CV_MAE_std"],
                "Train_R2": tt["Train_R2"], "Train_RMSE": tt["Train_RMSE"],
                "Train_MAE": tt["Train_MAE"],
                "Test_R2": tt["Test_R2"], "Test_RMSE": tt["Test_RMSE"],
                "Test_MAE": tt["Test_MAE"],
                "Best_parameters": tt["Best_parameters"],
            })

iyilesme_satirlari = []
for hedef in TARGETS:
    for model_adi in MODELLER:
        kisa = TARGET_KISA[hedef]
        al = lambda y, s: karsilastirma_df_gecici[(karsilastirma_df_gecici["Target"] == kisa) &
                                                  (karsilastirma_df_gecici["Model"] == model_adi) &
                                                  (karsilastirma_df_gecici["Optimization_method"] == y)][s].iloc[0]
        karsilastirma_df_gecici = pd.DataFrame(karsilastirma_satirlari)
        for kaynak, hedef_yontem in [("Baseline", "Randomized Search"),
                                     ("Baseline", "PSO"),
                                     ("Randomized Search", "PSO")]:
            iyilesme_satirlari.append({
                "Target": kisa, "Model": model_adi,
                "Comparison": f"{kaynak} -> {hedef_yontem}",
                "CV_RMSE_from": al(kaynak, "CV_RMSE_mean"),
                "CV_RMSE_to": al(hedef_yontem, "CV_RMSE_mean"),
                "CV_RMSE_improvement_%": iyilesme_yuzdesi(al(kaynak, "CV_RMSE_mean"),
                                                          al(hedef_yontem, "CV_RMSE_mean")),
                "Test_RMSE_from": al(kaynak, "Test_RMSE"),
                "Test_RMSE_to": al(hedef_yontem, "Test_RMSE"),
                "Test_RMSE_improvement_%": iyilesme_yuzdesi(al(kaynak, "Test_RMSE"),
                                                            al(hedef_yontem, "Test_RMSE")),
            })

karsilastirma_df = pd.DataFrame(karsilastirma_satirlari)
iyilesme_df = pd.DataFrame(iyilesme_satirlari)

# ---- Hesaplama maliyeti ----------------------------------------------
maliyet_df = pd.concat([pd.DataFrame(rs_maliyet), pd.DataFrame(pso_maliyet)],
                       ignore_index=True)
maliyet_df = maliyet_df.merge(
    karsilastirma_df[["Target", "Model", "Optimization_method",
                      "CV_RMSE_mean", "CV_RMSE_std", "Test_RMSE", "Test_MAE", "Test_R2"]],
    on=["Target", "Model", "Optimization_method"], how="left")

# ---- Excel cikti dosyalari -------------------------------------------
def sayfalara_yaz(writer, tablo, isim_fonksiyonu):
    """Tabloyu hedef ve modele gore ayri sayfalara yazar."""
    for hedef in TARGETS:
        for model_adi in MODELLER:
            alt = tablo[(tablo["Target"] == TARGET_KISA[hedef]) & (tablo["Model"] == model_adi)]
            alt.round(6).to_excel(writer, sheet_name=isim_fonksiyonu(hedef, model_adi),
                                  index=False)


sayfa_adi = lambda h, m: f"{TARGET_KISA[h]}_{m}"

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Optimized_Model_Parameters.xlsx"),
                    engine="openpyxl") as writer:
    tt_df[["Target", "Model", "Optimization_method", "Best_parameters"]].to_excel(
        writer, sheet_name="Best_parameters", index=False)
    kararlilik_df.round(6).to_excel(writer, sheet_name="PSO_runs", index=False)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Optimized_CV_Results.xlsx"),
                    engine="openpyxl") as writer:
    cv_ozet_df.round(6).to_excel(writer, sheet_name="CV_summary", index=False)
    sayfalara_yaz(writer, cv_fold_df, lambda h, m: f"folds_{TARGET_KISA[h]}_{m}")

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Optimized_Train_Test_Results.xlsx"),
                    engine="openpyxl") as writer:
    for hedef in TARGETS:
        tt_df[tt_df["Target"] == TARGET_KISA[hedef]].round(6).to_excel(
            writer, sheet_name=f"{TARGET_KISA[hedef]}_train_test", index=False)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Baseline_RandomSearch_PSO_Comparison.xlsx"),
                    engine="openpyxl") as writer:
    for hedef in TARGETS:
        karsilastirma_df[karsilastirma_df["Target"] == TARGET_KISA[hedef]].round(6).to_excel(
            writer, sheet_name=f"{TARGET_KISA[hedef]}_comparison", index=False)
    iyilesme_df.round(6).to_excel(writer, sheet_name="Improvements", index=False)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Optimization_Computational_Cost.xlsx"),
                    engine="openpyxl") as writer:
    maliyet_df.round(6).to_excel(writer, sheet_name="Cost_comparison", index=False)
    CV_KONTROL.to_excel(writer, sheet_name="CV_fold_check", index=False)
    pd.DataFrame([
        {"Setting": "RANDOM_SEARCH_N_ITER", "Value": RANDOM_SEARCH_N_ITER},
        {"Setting": "PSO_SWARM_SIZE", "Value": PSO_SWARM_SIZE},
        {"Setting": "PSO_N_ITERATIONS", "Value": PSO_N_ITERATIONS},
        {"Setting": "PSO_COUNT_INITIAL_AS_ITERATION", "Value": PSO_COUNT_INITIAL_AS_ITERATION},
        {"Setting": "RUN_MULTIPLE_PSO_RUNS", "Value": RUN_MULTIPLE_PSO_RUNS},
        {"Setting": "PSO_SEEDS", "Value": str(kullanilacak_seedler)},
        {"Setting": "Scoring", "Value": SCORING},
    ]).to_excel(writer, sheet_name="Settings", index=False)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Optimized_Test_Predictions.xlsx"),
                    engine="openpyxl") as writer:
    sayfalara_yaz(writer, tahmin_df, sayfa_adi)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Optimized_Range_Subgroup_Results.xlsx"),
                    engine="openpyxl") as writer:
    for hedef in TARGETS:
        altgrup_df[altgrup_df["Target"] == TARGET_KISA[hedef]].round(6).to_excel(
            writer, sheet_name=f"{TARGET_KISA[hedef]}_subgroups", index=False)
    pd.DataFrame({"Outside_range_geometries": sorted(bulunan_disarida)}).to_excel(
        writer, sheet_name="Outside_geometries", index=False)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Optimized_HighFrequency_Results.xlsx"),
                    engine="openpyxl") as writer:
    for hedef in TARGETS:
        yuksek_frekans_df[yuksek_frekans_df["Target"] == TARGET_KISA[hedef]].round(6).to_excel(
            writer, sheet_name=f"{TARGET_KISA[hedef]}_high_freq", index=False)
    pd.DataFrame([{"Note": "The 2.5 Hz threshold is used only as a diagnostic grouping. "
                           "It was not used in hyperparameter selection, optimization, "
                           "fitness weighting or sample weighting."}]).to_excel(
        writer, sheet_name="Notes", index=False)

print("\nTum Excel dosyalari olusturuldu.")

In [ ]:
# =====================================================================
# BOLUM 8 - Makale kalitesinde grafikler
# =====================================================================

HEDEF_ETIKET = {"f1": "$f_1$", "f2": "$f_2$"}
METRIK_ETIKET = {"CV_RMSE": "Cross-validation RMSE (Hz)", "Test_RMSE": "Test RMSE (Hz)",
                 "Test_MAE": "Test MAE (Hz)", "Test_R2": "Test $R^2$ (-)"}


def yontem_karsilastirma(hedef_kisa, model_adi, metrik):
    """Baseline / Randomized Search / PSO karsilastirma cubuk grafigi."""
    alt = karsilastirma_df[(karsilastirma_df["Target"] == hedef_kisa) &
                           (karsilastirma_df["Model"] == model_adi)]
    alt = alt.set_index("Optimization_method").loc[YONTEMLER].reset_index()

    if metrik == "CV_RMSE":
        deger, hata = alt["CV_RMSE_mean"].values, alt["CV_RMSE_std"].values
    else:
        deger, hata = alt[metrik].values, None

    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    ax.bar(alt["Optimization_method"], deger, yerr=hata, capsize=4,
           color=[RENK[y] for y in alt["Optimization_method"]],
           edgecolor="white", width=0.6,
           error_kw=dict(ecolor="#333333", lw=1.0))
    ax.set_ylabel(METRIK_ETIKET[metrik])
    ax.set_xlabel("Optimization method")
    ax.set_title(f"{model_adi} - Target: {HEDEF_ETIKET[hedef_kisa]}", loc="left")
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, "opt_comp", f"Fig_{metrik}_{hedef_kisa}_{model_adi}")


def optimize_avp(yontem, model_adi, hedef):
    """Optimize edilmis model icin actual vs predicted grafigi."""
    kisa = TARGET_KISA[hedef]
    pipe = egitilmis_modeller[(yontem, model_adi, hedef)]
    gercek = y_test[hedef].values
    tahmin = pipe.predict(X_test)
    m = metrik_seti(gercek, tahmin)

    fig, ax = plt.subplots(figsize=(5.4, 5.2))
    ic = ~disarida_mask
    ax.scatter(gercek[ic], tahmin[ic], s=26, color="#4C72B0", alpha=0.6,
               edgecolors="none", label="Inside training range")
    ax.scatter(gercek[disarida_mask], tahmin[disarida_mask], s=52, marker="^",
               facecolors="none", edgecolors=RENK_VURGU, linewidths=1.4,
               label="Outside training range")
    alt_s = min(gercek.min(), tahmin.min())
    ust_s = max(gercek.max(), tahmin.max())
    pay = 0.05 * (ust_s - alt_s)
    ax.plot([alt_s - pay, ust_s + pay], [alt_s - pay, ust_s + pay],
            color="#333333", lw=1.2, ls="--", label="y = x")
    ax.set_xlim(alt_s - pay, ust_s + pay)
    ax.set_ylim(alt_s - pay, ust_s + pay)
    ax.set_aspect("equal", adjustable="box")
    ax.text(0.04, 0.96,
            f"$R^2$ = {m['R2']:.3f}\nRMSE = {m['RMSE']:.4f} Hz\n"
            f"MAE = {m['MAE']:.4f} Hz\nMean error = {m['Mean_Error']:.4f} Hz",
            transform=ax.transAxes, va="top", ha="left", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white",
                      edgecolor="0.8", alpha=0.9))
    ax.set_xlabel(f"Actual {HEDEF_ETIKET[kisa]} (Hz)")
    ax.set_ylabel(f"Predicted {HEDEF_ETIKET[kisa]} (Hz)")
    ax.set_title(f"{model_adi} - {yontem}", loc="left")
    ax.legend(frameon=False, loc="lower right")
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, "avp", f"Fig_AVP_{kisa}_{model_adi}_{yontem.replace(' ', '')}")


def optimize_artik(yontem, model_adi, hedef):
    """Predicted vs residual grafigi; alt gruplar farkli isaretleyicilerle."""
    kisa = TARGET_KISA[hedef]
    pipe = egitilmis_modeller[(yontem, model_adi, hedef)]
    gercek = y_test[hedef].values
    tahmin = pipe.predict(X_test)
    artik = gercek - tahmin
    yuksek = yuksek_frekans_mask[hedef]

    fig, ax = plt.subplots(figsize=(5.8, 4.6))
    normal = (~disarida_mask) & (~yuksek)
    ax.scatter(tahmin[normal], artik[normal], s=26, color="#4C72B0", alpha=0.6,
               edgecolors="none", label="Inside range, $\\leq$ 2.5 Hz")
    ax.scatter(tahmin[(~disarida_mask) & yuksek], artik[(~disarida_mask) & yuksek],
               s=44, marker="s", facecolors="none", edgecolors="#333333",
               linewidths=1.2, label="Inside range, > 2.5 Hz")
    ax.scatter(tahmin[disarida_mask], artik[disarida_mask], s=52, marker="^",
               facecolors="none", edgecolors=RENK_VURGU, linewidths=1.4,
               label="Outside training range")
    ax.axhline(0, color="#333333", lw=1.1, ls="--")
    ax.set_xlabel(f"Predicted {HEDEF_ETIKET[kisa]} (Hz)")
    ax.set_ylabel(f"Residual (actual - predicted) {HEDEF_ETIKET[kisa]} (Hz)")
    ax.set_title(f"{model_adi} - {yontem}", loc="left")
    ax.legend(frameon=False, loc="best")
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, "resid", f"Fig_Residual_{kisa}_{model_adi}_{yontem.replace(' ', '')}")


def pso_yakinsama(model_adi, hedef):
    """Ana kosu icin global best ve ortalama suru yakinsama egrileri."""
    kisa = TARGET_KISA[hedef]
    gecmis = pso_kosulari[(model_adi, hedef, RANDOM_STATE)]["history"]
    fig, ax = plt.subplots(figsize=(6.0, 4.4))
    ax.plot(gecmis["Iteration"], gecmis["Global_best_CV_RMSE"], marker="o",
            color="#4C72B0", lw=1.8, label="Global best")
    ax.plot(gecmis["Iteration"], gecmis["Mean_swarm_CV_RMSE"], marker="s",
            color="#8C8C8C", lw=1.4, ls="--", label="Mean swarm")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Cross-validation RMSE (Hz)")
    ax.set_title(f"{model_adi} - Target: {HEDEF_ETIKET[kisa]} (seed {RANDOM_STATE})", loc="left")
    ax.legend(frameon=False)
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, "pso_conv", f"Fig_PSO_convergence_{kisa}_{model_adi}")


def pso_kararlilik(model_adi, hedef):
    """Tum seed'ler icin yakinsama egrileri, ortalama ve standart sapma bandi."""
    kisa = TARGET_KISA[hedef]
    fig, ax = plt.subplots(figsize=(6.2, 4.4))
    matris = []
    for seed in kullanilacak_seedler:
        g = pso_kosulari[(model_adi, hedef, seed)]["history"]
        ax.plot(g["Iteration"], g["Global_best_CV_RMSE"], lw=1.0, alpha=0.55,
                color="#8C8C8C", label="Individual runs" if seed == kullanilacak_seedler[0] else None)
        matris.append(g["Global_best_CV_RMSE"].values)

    matris = np.vstack(matris)
    iterasyonlar = np.arange(matris.shape[1])
    ortalama, std = matris.mean(axis=0), matris.std(axis=0, ddof=1) if matris.shape[0] > 1 else np.zeros(matris.shape[1])
    ax.plot(iterasyonlar, ortalama, color="#4C72B0", lw=2.0, marker="o", label="Mean of runs")
    ax.fill_between(iterasyonlar, ortalama - std, ortalama + std, color="#4C72B0",
                    alpha=0.18, label="$\\pm$ 1 std")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Global best cross-validation RMSE (Hz)")
    ax.set_title(f"{model_adi} - Target: {HEDEF_ETIKET[kisa]}", loc="left")
    ax.legend(frameon=False)
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, "pso_stab", f"Fig_PSO_stability_{kisa}_{model_adi}")


def rs_parametre_grafigi(model_adi, hedef, param, log_eksen=True):
    """Randomized Search adaylarinin parametre - CV RMSE dagilimi."""
    kisa = TARGET_KISA[hedef]
    alt = rs_tum[(rs_tum["Target"] == kisa) & (rs_tum["Model"] == model_adi)]
    fig, ax = plt.subplots(figsize=(5.8, 4.4))
    ax.scatter(alt[param], alt["Mean_validation_RMSE"], s=28, color="#4C72B0",
               alpha=0.65, edgecolors="none")
    if log_eksen:
        ax.set_xscale("log")
    ax.set_xlabel(param.replace("model__", ""))
    ax.set_ylabel("Mean cross-validation RMSE (Hz)")
    ax.set_title(f"{model_adi} - Target: {HEDEF_ETIKET[kisa]}", loc="left")
    ax.text(0.02, 0.02, "Other hyperparameters vary simultaneously;\n"
                        "this plot does not show a univariate causal effect.",
            transform=ax.transAxes, fontsize=8, va="bottom", ha="left", color="#555555")
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, "rs_analiz", f"Fig_RS_{kisa}_{model_adi}_{param.replace('model__', '')}")


def rs_max_depth_grafigi(hedef):
    """GBR icin max_depth gruplarina gore CV RMSE kutu grafigi."""
    kisa = TARGET_KISA[hedef]
    alt = rs_tum[(rs_tum["Target"] == kisa) & (rs_tum["Model"] == "GBR")]
    fig, ax = plt.subplots(figsize=(5.6, 4.4))
    gruplar = sorted(alt["model__max_depth"].dropna().unique())
    ax.boxplot([alt.loc[alt["model__max_depth"] == g, "Mean_validation_RMSE"].values
                for g in gruplar], labels=[int(g) for g in gruplar], widths=0.5)
    ax.set_xlabel("max_depth")
    ax.set_ylabel("Mean cross-validation RMSE (Hz)")
    ax.set_title(f"GBR - Target: {HEDEF_ETIKET[kisa]}", loc="left")
    ax.text(0.02, 0.02, "Other hyperparameters vary simultaneously.",
            transform=ax.transAxes, fontsize=8, va="bottom", ha="left", color="#555555")
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, "rs_analiz", f"Fig_RS_{kisa}_GBR_max_depth_box")


def maliyet_grafigi(model_adi, hedef, sutun, y_etiket, dosya_soneki):
    """Optimizasyon yontemi ile maliyet/performans karsilastirmasi."""
    kisa = TARGET_KISA[hedef]
    alt = maliyet_df[(maliyet_df["Target"] == kisa) & (maliyet_df["Model"] == model_adi)]
    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    ax.bar(alt["Optimization_method"], alt[sutun].values,
           color=[RENK[y] for y in alt["Optimization_method"]],
           edgecolor="white", width=0.55)
    ax.set_ylabel(y_etiket)
    ax.set_xlabel("Optimization method")
    ax.set_title(f"{model_adi} - Target: {HEDEF_ETIKET[kisa]}", loc="left")
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, "cost", f"Fig_Cost_{kisa}_{model_adi}_{dosya_soneki}")


print("\n--- Grafikler uretiliyor ---")
for hedef in TARGETS:
    kisa = TARGET_KISA[hedef]
    for model_adi in MODELLER:
        for metrik in ["CV_RMSE", "Test_RMSE", "Test_MAE", "Test_R2"]:
            yontem_karsilastirma(kisa, model_adi, metrik)
        for yontem in ["Randomized Search", "PSO"]:
            optimize_avp(yontem, model_adi, hedef)
            optimize_artik(yontem, model_adi, hedef)
        pso_yakinsama(model_adi, hedef)
        if len(kullanilacak_seedler) > 1:
            pso_kararlilik(model_adi, hedef)
        maliyet_grafigi(model_adi, hedef, "CV_RMSE_mean",
                        "Best cross-validation RMSE (Hz)", "best_cv_rmse")
        maliyet_grafigi(model_adi, hedef, "Runtime_seconds", "Runtime (s)", "runtime")
        maliyet_grafigi(model_adi, hedef, "Total_model_fits", "Total model fits", "fits")

    for param in ["model__C", "model__gamma", "model__epsilon"]:
        rs_parametre_grafigi("SVR", hedef, param, log_eksen=True)
    rs_parametre_grafigi("GBR", hedef, "model__learning_rate", log_eksen=True)
    rs_parametre_grafigi("GBR", hedef, "model__n_estimators", log_eksen=False)
    rs_max_depth_grafigi(hedef)

print("\nAsama 5 tamamlandi. Nihai model secilmedi, SHAP veya ozellik onem analizi yapilmadi.")
print("Cikti klasoru:", OUTPUT_DIR)